# 🤖 Week 4: Supervised Machine Learning with NumPy

Welcome to the final week! You will build real machine learning models **from scratch using only NumPy**, then test them on a real **Kaggle** dataset. No `scikit-learn` yet. Building it yourself is the best way to understand what is happening inside the "magic" libraries.

Everything you learned is used here: functions and classes (Weeks 1–2), Git for saving your progress (Week 3), and NumPy arrays, `axis`, broadcasting, masks and `nan` handling (Week 3).

**Time needed:** about 5 days, 1–2 hours a day.

**📖 Official docs and learning resources:**
- [Kaggle Learn: Intro to Machine Learning](https://www.kaggle.com/learn/intro-to-machine-learning) (free, short)
- [NumPy linear algebra (`numpy.linalg`)](https://numpy.org/doc/stable/reference/routines.linalg.html)
- [scikit-learn User Guide](https://scikit-learn.org/stable/user_guide.html): read the *Linear models* and *Nearest Neighbors* intro to compare with what you build
- [Kaggle Pima Indians Diabetes dataset page](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)

### Plan for the week
| Day | Topics |
|---|---|
| 1 | What is supervised learning? Get the data, explore, clean |
| 2 | Train/test split and feature scaling |
| 3 | Linear regression (predict a number) |
| 4 | Logistic regression (predict a category) |
| 5 | k-Nearest Neighbors, evaluation metrics, compare models, final project |

> ⚠️ **Do the exercises in order.** Later cells use the functions you write earlier. If you restart the notebook, use *Run → Run All Above* to restore your functions.

In [ ]:
# Run this cell first! It sets up a small helper that checks your exercise answers.
def check(test, hint=""):
    """Runs test() and prints a friendly pass / not-yet message."""
    try:
        ok = bool(test())
    except Exception as err:
        ok = False
        hint = f"{hint} (Python said: {type(err).__name__}: {err})"
    print("✅ Correct! Nice work." if ok else f"❌ Not yet. {hint}")

print("Setup done. You are ready to go!")
import numpy as np

## Day 1: What is supervised learning?

In **supervised learning** we show the computer examples **with the correct answers**, and it learns the pattern.

| Word | Meaning | In our project |
|---|---|---|
| **Features** (`X`) | the inputs, a table of numbers | glucose, BMI, age... |
| **Label / target** (`y`) | the answer we want to predict | has diabetes? (0 or 1) |
| **Training** | the model learns from examples | find the best weights |
| **Prediction** | guess `y` for new `X` | is this new patient at risk? |

Two main types:
- **Regression**: predict a **number** (house price, BMI). Example: linear regression.
- **Classification**: predict a **category** (spam / not spam, diabetes / no diabetes). Examples: logistic regression, k-NN.

The big rule: **never test a model on the same data it learned from.** We keep some data hidden (the *test set*) to measure how well it works on new data.

### Step 1: Get the Kaggle dataset
1. Create a free account at [kaggle.com](https://www.kaggle.com).
2. Open the [Pima Indians Diabetes Database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) page and click **Download**.
3. Unzip it and put **`diabetes.csv`** in this repository's **`data/`** folder, so the path is `data/diabetes.csv`.

The data has 768 patients. Each row has 8 measurements and an `Outcome` (1 = has diabetes, 0 = no).

| Column | Meaning |
|---|---|
| Pregnancies | number of pregnancies |
| Glucose | blood sugar level |
| BloodPressure | blood pressure (mm Hg) |
| SkinThickness | skin-fold thickness (mm) |
| Insulin | insulin level |
| BMI | body mass index |
| DiabetesPedigreeFunction | family history score |
| Age | age in years |
| **Outcome** | **0 / 1 (our label)** |

In [ ]:
import os

DATA_PATH = os.path.join("..", "data", "diabetes.csv")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "Could not find data/diabetes.csv. Download it from Kaggle (see the steps above) "
        "and put it in the 'data' folder of this repository, then run this cell again."
    )

with open(DATA_PATH) as f:
    column_names = f.readline().strip().split(",")

data = np.genfromtxt(DATA_PATH, delimiter=",", skip_header=1)
print("Columns:", column_names)
print("Data shape (rows, columns):", data.shape)
print(data[:3])

### Split into features and label
All columns except the last are features `X`; the last column is the label `y`. (Slicing from Week 3!)

In [ ]:
X_raw = data[:, :-1]            # every row, all columns except the last
y = data[:, -1].astype(int)     # last column as whole numbers 0/1
feature_names = column_names[:-1]

print("X shape:", X_raw.shape, " y shape:", y.shape)
print("People with diabetes:", y.sum(), "out of", len(y))
print("Share with diabetes:", round(y.mean(), 3))

A model that always says "no diabetes" would already be right about **65%** of the time. That is our **baseline**. A useful model must beat it, and we must look at more than accuracy later.

### Explore the data
Look at the minimum of each column. Do you see anything strange?

In [ ]:
print(f"{'feature':26s}{'min':>8s}{'mean':>9s}{'max':>8s}")
for i, name in enumerate(feature_names):
    col = X_raw[:, i]
    print(f"{name:26s}{col.min():8.1f}{col.mean():9.1f}{col.max():8.1f}")

👀 **Found it?** `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` and `BMI` have a minimum of **0**. A person cannot have 0 blood pressure, so these zeros really mean **missing data**. If we leave them, the model learns from wrong numbers.

**Fix:** turn those zeros into `np.nan` and fill them with the **median** of the column (the median is less affected by extreme values than the mean). You practised `nan` handling in Week 3!

### 🏋️ Exercise 1: Clean the data

Write `clean_data(X, columns)`:
- Make a copy of `X` (don't change the original).
- For each column index in `columns`: treat `0` as missing, and replace those zeros with the median of that column's **non-zero** values.
- Return the cleaned copy.

Hint: `col = X[:, c]`, `missing = col == 0`, `np.median(col[~missing])`.

In [ ]:
def clean_data(X, columns):
    # YOUR CODE HERE
    pass

# ---- self-check (don't edit) ----
def _test():
    X = np.array([[0.0, 5.0], [10.0, 0.0], [20.0, 7.0], [0.0, 9.0], [30.0, 8.0]])
    out = clean_data(X, [0])
    # column 0 non-zero values: 10, 20, 30 -> median 20; column 1 untouched; original untouched
    return (out[:, 0].tolist() == [20, 10, 20, 20, 30] and out[1, 1] == 0 and X[0, 0] == 0)
check(_test, 'Zeros in the listed columns -> median of that column\'s non-zero values. Other columns and the original array stay unchanged')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
def clean_data(X, columns):
    X = X.copy()
    for c in columns:
        col = X[:, c]
        missing = col == 0
        col[missing] = np.median(col[~missing])
    return X
```

</details>

In [ ]:
ZERO_MEANS_MISSING = [1, 2, 3, 4, 5]   # Glucose, BloodPressure, SkinThickness, Insulin, BMI
X_clean = clean_data(X_raw, ZERO_MEANS_MISSING)

print("Minimum of each column after cleaning:")
print(dict(zip(feature_names, X_clean.min(axis=0).round(1))))

## Day 2: Train/test split and feature scaling

### Train/test split
We shuffle the rows, keep 80% to **train** and hide 20% to **test**. Shuffle by shuffling the *indexes* so `X` and `y` stay matched.

Use a fixed `seed` so you get the same split every time (reproducible).

### 🏋️ Exercise 2: Train/test split

Write `train_test_split(X, y, test_size=0.2, seed=42)` that returns `X_train, X_test, y_train, y_test`.

Steps: `rng = np.random.default_rng(seed)` -> `idx = rng.permutation(len(X))` -> the first `int(len(X) * test_size)` indexes are the test rows, the rest are training rows.

In [ ]:
def train_test_split(X, y, test_size=0.2, seed=42):
    # YOUR CODE HERE
    pass

# ---- self-check (don't edit) ----
def _test():
    X = np.arange(20).reshape(10, 2)
    y = X[:, 0] // 2
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, seed=1)
    rows_ok = len(Xtr) == 8 and len(Xte) == 2 and len(ytr) == 8 and len(yte) == 2
    matched = (Xtr[:, 0] // 2 == ytr).all() and (Xte[:, 0] // 2 == yte).all()
    no_overlap = len(set(Xtr[:, 0]) & set(Xte[:, 0])) == 0
    same_again = (train_test_split(X, y, 0.2, 1)[0] == Xtr).all()
    return rows_ok and matched and no_overlap and same_again
check(_test, '10 rows, 0.2 -> 8 train + 2 test, X and y rows must stay matched, no overlap, and the same seed gives the same split')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
def train_test_split(X, y, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X))
    n_test = int(len(X) * test_size)
    test_idx, train_idx = idx[:n_test], idx[n_test:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]
```

</details>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_clean, y, test_size=0.2, seed=42)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Diabetes share  train:", y_train.mean().round(3), " test:", y_test.mean().round(3))

### Feature scaling (standardization)
`Insulin` goes up to ~800 but `DiabetesPedigreeFunction` is below 3. Models that use distances or gradient descent get confused when features have very different sizes. **Standardize**: subtract the mean and divide by the standard deviation, so each feature has mean 0 and std 1.

$$x_{scaled} = \frac{x - \text{mean}}{\text{std}}$$

⚠️ **Important:** compute the mean and std from the **training data only**, then apply the same numbers to the test data. The test set is "the future"; using its numbers would be cheating (called *data leakage*).

### 🏋️ Exercise 3: Standardize

Write `standardize(X_train, X_test)` that returns `(X_train_scaled, X_test_scaled)` using the mean and std **of `X_train`** (per column, `axis=0`). If a column's std is 0, use 1 for it to avoid dividing by zero.

In [ ]:
def standardize(X_train, X_test):
    # YOUR CODE HERE
    pass

# ---- self-check (don't edit) ----
def _test():
    tr = np.array([[1.0, 5.0], [3.0, 5.0], [5.0, 5.0]])
    te = np.array([[7.0, 5.0]])
    a, b = standardize(tr, te)
    std0 = tr[:, 0].std()
    return (np.allclose(a.mean(axis=0), 0) and np.isclose(a[:, 0].std(), 1)
            and np.allclose(b, [[(7 - 3) / std0, 0]]))
check(_test, 'Train columns get mean 0 / std 1; test uses TRAIN mean and std; a constant column should not cause a divide-by-zero')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
def standardize(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1
    return (X_train - mean) / std, (X_test - mean) / std
```

</details>

In [ ]:
Xs_train, Xs_test = standardize(X_train, X_test)
print("Train means (should be ~0):", Xs_train.mean(axis=0).round(2))
print("Train stds  (should be ~1):", Xs_train.std(axis=0).round(2))

## Day 3: Linear regression: predicting a number

Idea: draw the best straight line through the points.

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots$$

- `w` are the **weights** (what the model learns). `w_0` is the *bias* (intercept).
- "Best" means the smallest **MSE** (mean squared error): the average of (prediction − truth)².

In matrix form we add a column of ones to `X` (for the bias), then `ŷ = X_b @ w` (`@` is matrix multiplication). There are two ways to find `w`:
1. **Normal equation**: solve directly with algebra: `w = (XᵀX)⁻¹ Xᵀ y`
2. **Gradient descent**: start with `w = 0` and take many small steps downhill on the error.

📖 Docs: [`np.linalg.solve`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html), [`@` / `np.matmul`](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html)

Let's start with a tiny made-up dataset: hours studied -> exam score.

In [ ]:
rng = np.random.default_rng(0)
hours = rng.uniform(0, 10, size=60)
score = 35 + 6 * hours + rng.normal(0, 5, size=60)    # true rule + noise

X_toy = hours.reshape(-1, 1)    # features must be 2D: (rows, features)
y_toy = score
print(X_toy.shape, y_toy.shape)

try:
    import matplotlib.pyplot as plt
    plt.scatter(hours, score)
    plt.xlabel("hours studied"); plt.ylabel("exam score"); plt.show()
except ImportError:
    print("(matplotlib not installed. Plots are optional)")

### 🏋️ Exercise 4: Normal equation, predict, MSE and R²

Write four functions:
1. `add_bias(X)` -> `X` with a column of ones added **as the first column** (`np.ones((len(X), 1))` and `np.hstack`).
2. `fit_normal(X, y)` -> weights `w` using `np.linalg.solve(Xb.T @ Xb, Xb.T @ y)` where `Xb = add_bias(X)`.
3. `predict_linear(X, w)` -> `add_bias(X) @ w`.
4. `mse(y_true, y_pred)` -> mean of squared errors, and `r2_score(y_true, y_pred)` -> `1 - SS_res / SS_tot` where `SS_res = sum((y - pred)²)`, `SS_tot = sum((y - mean(y))²)`. (R² = 1 is perfect, 0 is no better than guessing the average.)

In [ ]:
def add_bias(X):
    # YOUR CODE HERE
    pass

def fit_normal(X, y):
    # YOUR CODE HERE
    pass

def predict_linear(X, w):
    # YOUR CODE HERE
    pass

def mse(y_true, y_pred):
    # YOUR CODE HERE
    pass

def r2_score(y_true, y_pred):
    # YOUR CODE HERE
    pass

# ---- self-check (don't edit) ----
def _test():
    X = np.array([[1.0], [2.0], [3.0]])
    y = np.array([3.0, 5.0, 7.0])           # y = 1 + 2x
    w = fit_normal(X, y)
    return (add_bias(X).shape == (3, 2) and (add_bias(X)[:, 0] == 1).all()
            and np.allclose(w, [1, 2]) and np.allclose(predict_linear(np.array([[10.0]]), w), [21])
            and np.isclose(mse(np.array([1.0, 2.0]), np.array([2.0, 4.0])), 2.5)
            and np.isclose(r2_score(y, predict_linear(X, w)), 1.0)
            and np.isclose(r2_score(y, np.full(3, y.mean())), 0.0))
check(_test, 'y = 1 + 2x should give w = [1, 2]. Bias column must come FIRST. MSE of errors (1, 2) is 2.5')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
def add_bias(X):
    return np.hstack([np.ones((len(X), 1)), X])

def fit_normal(X, y):
    Xb = add_bias(X)
    return np.linalg.solve(Xb.T @ Xb, Xb.T @ y)

def predict_linear(X, w):
    return add_bias(X) @ w

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot
```

</details>

In [ ]:
w = fit_normal(X_toy, y_toy)
pred = predict_linear(X_toy, w)
print("Learned weights [bias, slope]:", w.round(2), " (true values were about [35, 6])")
print("MSE:", mse(y_toy, pred).round(2), " R²:", r2_score(y_toy, pred).round(3))

### Gradient descent
Instead of solving algebra, walk downhill. Each step:

1. predictions `ŷ = X_b @ w`
2. gradient `= (2 / n) · X_bᵀ @ (ŷ − y)` (the slope of the error)
3. `w = w − learning_rate · gradient`

The **learning rate** is the step size. Too big and you overshoot, too small and it takes forever. This same idea trains giant neural networks!

### 🏋️ Exercise 5: Gradient descent

Write `fit_gd(X, y, lr=0.01, epochs=1000)` that returns the weights after `epochs` steps, starting from all zeros. (One *epoch* = one update of the weights.)

In [ ]:
def fit_gd(X, y, lr=0.01, epochs=1000):
    # YOUR CODE HERE
    pass

# ---- self-check (don't edit) ----
def _test():
    X = np.array([[1.0], [2.0], [3.0]])
    y = np.array([3.0, 5.0, 7.0])
    w = fit_gd(X, y, lr=0.05, epochs=5000)
    return np.allclose(w, [1, 2], atol=0.05)
check(_test, 'On y = 1 + 2x with lr=0.05 and 5000 epochs, the weights should end close to [1, 2]')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
def fit_gd(X, y, lr=0.01, epochs=1000):
    Xb = add_bias(X)
    n = len(Xb)
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        gradient = (2 / n) * Xb.T @ (Xb @ w - y)
        w = w - lr * gradient
    return w
```

</details>

In [ ]:
w_gd = fit_gd(X_toy, y_toy, lr=0.01, epochs=3000)
print("Normal equation:", w.round(2))
print("Gradient descent:", w_gd.round(2))

**Try it:** set `lr=0.2` and see what happens (does it blow up?). Then try `lr=0.0001`. This is what *tuning* means.

### Real data: predict BMI from other body measurements
Now use the cleaned Pima data. We treat BMI as the number to predict (a regression task) from glucose, blood pressure, skin thickness, insulin and age.

In [ ]:
reg_inputs = [1, 2, 3, 4, 7]    # Glucose, BloodPressure, SkinThickness, Insulin, Age
reg_target = 5                  # BMI

Xr_train, Xr_test = Xs_train[:, reg_inputs], Xs_test[:, reg_inputs]
yr_train, yr_test = X_train[:, reg_target], X_test[:, reg_target]   # raw BMI values

w_bmi = fit_normal(Xr_train, yr_train)
print("Weights (bias first):", w_bmi.round(2))
print("Train R²:", r2_score(yr_train, predict_linear(Xr_train, w_bmi)).round(3))
print("Test  R²:", r2_score(yr_test,  predict_linear(Xr_test,  w_bmi)).round(3))
print("Test RMSE (average BMI error):", np.sqrt(mse(yr_test, predict_linear(Xr_test, w_bmi))).round(2))

💭 **Think:** the R² is modest. Skin thickness predicts BMI quite well, but the other inputs add little. Linear regression is only as good as the information in the features. Check which weight is biggest, and compare with the `feature_names`.

## Day 4: Logistic regression: predicting a category

For a yes/no answer (diabetes or not) we cannot use a straight line, because it could predict −3 or 7. Logistic regression squeezes the line's output into **0 to 1** (a probability) with the **sigmoid** function:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

- `z = X_b @ w` (same as before)
- `p = sigmoid(z)` = probability of class 1
- predict class 1 if `p >= 0.5`

The error we minimise is **log loss** (binary cross-entropy):

$$L = -\text{mean}\big(y \log p + (1-y)\log(1-p)\big)$$

Good news: the gradient looks almost identical to linear regression: `X_bᵀ @ (p − y) / n`.

### 🏋️ Exercise 6: Sigmoid, loss and prediction helpers

Write:
1. `sigmoid(z)` -> `1 / (1 + np.exp(-z))` (works on whole arrays). Clip `z` to `[-500, 500]` first with `np.clip` to avoid overflow.
2. `log_loss(y, p)` -> the formula above. Clip `p` into `[1e-9, 1 - 1e-9]` so `log(0)` never happens.
3. `predict_proba(X, w)` -> `sigmoid(add_bias(X) @ w)`.
4. `predict_class(X, w, threshold=0.5)` -> array of 0/1 integers.

In [ ]:
def sigmoid(z):
    # YOUR CODE HERE
    pass

def log_loss(y, p):
    # YOUR CODE HERE
    pass

def predict_proba(X, w):
    # YOUR CODE HERE
    pass

def predict_class(X, w, threshold=0.5):
    # YOUR CODE HERE
    pass

# ---- self-check (don't edit) ----
def _test():
    return (np.isclose(sigmoid(0), 0.5) and sigmoid(100) > 0.999 and sigmoid(-100) < 0.001
            and np.allclose(sigmoid(np.array([0.0, 0.0])), [0.5, 0.5])
            and np.isclose(log_loss(np.array([1, 0]), np.array([0.5, 0.5])), np.log(2))
            and np.isfinite(log_loss(np.array([1]), np.array([0.0])))
            and predict_class(np.array([[2.0], [-2.0]]), np.array([0.0, 1.0])).tolist() == [1, 0])
check(_test, 'sigmoid(0)=0.5; log_loss for p=0.5 is ln(2); positive z -> class 1')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def log_loss(y, p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def predict_proba(X, w):
    return sigmoid(add_bias(X) @ w)

def predict_class(X, w, threshold=0.5):
    return (predict_proba(X, w) >= threshold).astype(int)
```

</details>

### 🏋️ Exercise 7: Train logistic regression

Write `fit_logistic(X, y, lr=0.1, epochs=1000)` that returns `(w, losses)`:
- start `w` as zeros (length = number of features + 1)
- each epoch: `p = predict_proba(X, w)`, `gradient = add_bias(X).T @ (p - y) / len(X)`, `w = w - lr * gradient`
- save `log_loss(y, p)` into the `losses` list every epoch (so we can plot learning!).

In [ ]:
def fit_logistic(X, y, lr=0.1, epochs=1000):
    # YOUR CODE HERE
    pass

# ---- self-check (don't edit) ----
def _test():
    X = np.array([[-2.0], [-1.0], [1.0], [2.0]])
    y = np.array([0, 0, 1, 1])
    w, losses = fit_logistic(X, y, lr=0.5, epochs=500)
    return (len(losses) == 500 and losses[-1] < losses[0]
            and predict_class(X, w).tolist() == [0, 0, 1, 1])
check(_test, 'Loss should go down over the epochs and the 4 toy points should be classified perfectly')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
def fit_logistic(X, y, lr=0.1, epochs=1000):
    Xb = add_bias(X)
    w = np.zeros(Xb.shape[1])
    losses = []
    for _ in range(epochs):
        p = sigmoid(Xb @ w)
        losses.append(log_loss(y, p))
        w = w - lr * (Xb.T @ (p - y) / len(X))
    return w, losses
```

</details>

In [ ]:
w_log, losses = fit_logistic(Xs_train, y_train, lr=0.1, epochs=1000)
print("Loss at start:", round(losses[0], 4), " -> at end:", round(losses[-1], 4))

try:
    import matplotlib.pyplot as plt
    plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("log loss"); plt.title("The model is learning"); plt.show()
except ImportError:
    pass

train_acc = (predict_class(Xs_train, w_log) == y_train).mean()
test_acc = (predict_class(Xs_test, w_log) == y_test).mean()
print(f"Accuracy  train: {train_acc:.3f}   test: {test_acc:.3f}   (baseline 'always 0' is about 0.65)")

### Which features matter?
Because we standardized the features, the bigger a weight is (positive or negative), the more that feature influences the prediction.

In [ ]:
weights = w_log[1:]    # skip the bias
for i in np.argsort(-np.abs(weights)):
    print(f"{feature_names[i]:26s}{weights[i]:+.2f}")

## Day 5: k-Nearest Neighbors, evaluation, and comparing models

### k-Nearest Neighbors (k-NN)
The simplest idea in ML: **to classify a new point, look at the `k` most similar training points and let them vote.** There is no training step. It just remembers the data.

Similarity = **Euclidean distance**: `sqrt(sum((a - b)²))`. With broadcasting we can compute *all* distances without loops:
`X_test[:, None, :] - X_train[None, :, :]` has shape `(n_test, n_train, n_features)`.

### 🏋️ Exercise 8: k-NN classifier

Write `predict_knn(X_train, y_train, X_test, k=5)` that returns a 0/1 array:
1. `diff = X_test[:, None, :] - X_train[None, :, :]`
2. `dist = np.sqrt((diff ** 2).sum(axis=2))` -> shape `(n_test, n_train)`
3. `nearest = np.argsort(dist, axis=1)[:, :k]` -> indexes of the k closest training rows
4. `votes = y_train[nearest].mean(axis=1)` -> share of neighbours with label 1
5. predict 1 where `votes >= 0.5`.

In [ ]:
def predict_knn(X_train, y_train, X_test, k=5):
    # YOUR CODE HERE
    pass

# ---- self-check (don't edit) ----
def _test():
    Xt = np.array([[0.0], [1.0], [10.0], [11.0]])
    yt = np.array([0, 0, 1, 1])
    out = predict_knn(Xt, yt, np.array([[0.5], [10.5]]), k=3)
    return out.tolist() == [0, 1]
check(_test, 'A point near 0.5 should be class 0 and a point near 10.5 should be class 1 (k=3)')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
def predict_knn(X_train, y_train, X_test, k=5):
    diff = X_test[:, None, :] - X_train[None, :, :]
    dist = np.sqrt((diff ** 2).sum(axis=2))
    nearest = np.argsort(dist, axis=1)[:, :k]
    votes = y_train[nearest].mean(axis=1)
    return (votes >= 0.5).astype(int)
```

</details>

In [ ]:
knn_pred = predict_knn(Xs_train, y_train, Xs_test, k=5)
print("k-NN test accuracy:", (knn_pred == y_test).mean().round(3))

### Evaluation: accuracy is not enough
Imagine a test where only 1 in 100 people is sick. A model that always says "healthy" is 99% accurate, yet useless! For medical questions, we care *what kind* of mistakes the model makes. Count them in a **confusion matrix**:

| | Predicted 1 | Predicted 0 |
|---|---|---|
| **Actually 1** | TP (true positive) | FN (false negative: missed a sick person!) |
| **Actually 0** | FP (false positive: false alarm) | TN (true negative) |

- **Accuracy** = (TP + TN) / total
- **Precision** = TP / (TP + FP): when the model says "sick", how often is it right?
- **Recall** = TP / (TP + FN): of all truly sick people, how many did we catch?
- **F1** = `2 · precision · recall / (precision + recall)`: one number balancing both.

### 🏋️ Exercise 9: Classification metrics

Write `classification_report(y_true, y_pred)` returning a dict with keys `tp, fp, tn, fn, accuracy, precision, recall, f1`. If a denominator would be 0, return `0.0` for that metric. Hint: `((y_true == 1) & (y_pred == 1)).sum()` counts true positives.

In [ ]:
def classification_report(y_true, y_pred):
    # YOUR CODE HERE
    pass

# ---- self-check (don't edit) ----
def _test():
    yt = np.array([1, 1, 1, 0, 0, 0, 0, 1])
    yp = np.array([1, 0, 0, 0, 0, 1, 0, 1])
    r = classification_report(yt, yp)
    allzero = classification_report(np.array([1, 0]), np.array([0, 0]))
    return (r['tp'] == 2 and r['fn'] == 2 and r['fp'] == 1 and r['tn'] == 3
            and np.isclose(r['accuracy'], 5 / 8) and np.isclose(r['precision'], 2 / 3)
            and np.isclose(r['recall'], 0.5) and np.isclose(r['f1'], 4 / 7)
            and allzero['precision'] == 0.0 and allzero['f1'] == 0.0)
check(_test, 'Example: TP=2, FN=2, FP=1, TN=3 -> accuracy 0.625, precision 0.667, recall 0.5, f1 0.571. No crash when a denominator is 0')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
def classification_report(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn, 'accuracy': accuracy,
            'precision': precision, 'recall': recall, 'f1': f1}
```

</details>

### Compare all models on the test set

In [ ]:
baseline_pred = np.zeros_like(y_test)                       # always say "no diabetes"
log_pred = predict_class(Xs_test, w_log)
models = {"Baseline (always 0)": baseline_pred, "Logistic regression": log_pred, "k-NN (k=5)": knn_pred}

print(f"{'model':22s}{'accuracy':>10s}{'precision':>11s}{'recall':>9s}{'f1':>7s}")
for name, pred in models.items():
    r = classification_report(y_test, pred)
    print(f"{name:22s}{r['accuracy']:10.3f}{r['precision']:11.3f}{r['recall']:9.3f}{r['f1']:7.3f}")

print()
r = classification_report(y_test, log_pred)
print("Logistic regression confusion matrix:")
print(f"  caught sick: {r['tp']}   missed sick: {r['fn']}   false alarms: {r['fp']}   correctly healthy: {r['tn']}")

💭 **Think:** the baseline has good accuracy but a recall of 0: it never finds a sick patient. This is why we look at several numbers, not only accuracy.

### Overfitting: when a model memorises
A model that is **too flexible** can memorise the training data and then fail on new data. k-NN with `k=1` is a classic example: the closest point to a training row is itself, so training accuracy is 100%.

In [ ]:
print(f"{'k':>3s}{'train acc':>11s}{'test acc':>10s}")
for k in [1, 3, 5, 11, 21, 41]:
    tr = (predict_knn(Xs_train, y_train, Xs_train, k) == y_train).mean()
    te = (predict_knn(Xs_train, y_train, Xs_test, k) == y_test).mean()
    print(f"{k:3d}{tr:11.3f}{te:10.3f}")

Look at `k=1`: perfect on training, worse on test. That gap between train and test is the sign of overfitting.

> 📝 We used the test set to compare several `k` values, which is a small form of cheating. In real projects we keep a third set (the *validation* set) for choosing settings like `k`, and touch the test set only once at the end. You'll meet this in the scikit-learn docs as *cross-validation*.

## 🚀 Final project: build your own scikit-learn style model
Libraries like scikit-learn give every model the same interface: `fit`, `predict`, `score`. You learned classes in Week 2, so now wrap your logistic regression into a class.

Create `LogisticRegressionNumpy`:
- `__init__(self, lr=0.1, epochs=1000)`
- `fit(self, X, y)` -> trains with your `fit_logistic`, stores `self.weights` and `self.losses`, returns `self`
- `predict_proba(self, X)`, `predict(self, X, threshold=0.5)`
- `score(self, X, y)` -> accuracy

### 🏋️ Exercise 10: A model class

Complete the class below. Reuse the functions you already wrote.

In [ ]:
class LogisticRegressionNumpy:
    # YOUR CODE HERE
    pass

# ---- self-check (don't edit) ----
def _test():
    X = np.array([[-2.0], [-1.0], [1.0], [2.0]])
    y = np.array([0, 0, 1, 1])
    model = LogisticRegressionNumpy(lr=0.5, epochs=300).fit(X, y)
    return (model.score(X, y) == 1.0 and model.predict(np.array([[3.0]])).tolist() == [1]
            and len(model.losses) == 300 and model.predict_proba(np.array([[0.0]]))[0] > 0.4)
check(_test, 'fit() must return self, store weights and losses; score() is accuracy')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
class LogisticRegressionNumpy:
    def __init__(self, lr=0.1, epochs=1000):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.losses = []

    def fit(self, X, y):
        self.weights, self.losses = fit_logistic(X, y, self.lr, self.epochs)
        return self

    def predict_proba(self, X):
        return predict_proba(X, self.weights)

    def predict(self, X, threshold=0.5):
        return predict_class(X, self.weights, threshold)

    def score(self, X, y):
        return (self.predict(X) == y).mean()
```

</details>

In [ ]:
model = LogisticRegressionNumpy(lr=0.1, epochs=1000).fit(Xs_train, y_train)
print("Train accuracy:", round(model.score(Xs_train, y_train), 3))
print("Test accuracy: ", round(model.score(Xs_test, y_test), 3))

### 🏁 Challenges (no self-check. Explore and write what you find!)
Add new code cells below and try:
1. **Lower the threshold** to `0.3` in `predict_class`. What happens to recall and precision? When might a doctor prefer this?
2. **Feature selection:** train using only `Glucose`, `BMI` and `Age`. Is the test accuracy much worse?
3. **Learning rate lab:** train with `lr` = 0.001, 0.1, 1.0 and plot the losses. Which one learns best?
4. **Different split:** change `seed=42` to another number. How much does the test accuracy change? What does that tell you about trusting a single number?
5. **Compare with scikit-learn:** install it (`pip install scikit-learn`), run `LogisticRegression()` and `KNeighborsClassifier()` on the same data. Are your results similar? Read their docs to see what else they offer.
6. **Another Kaggle dataset:** try the same pipeline on a different small CSV from [Kaggle Datasets](https://www.kaggle.com/datasets).

When you are done, commit your work with Git (`git add .`, `git commit -m "Finish week 4"`) and push it to GitHub. 🎉

## ✅ Week 4 checklist
- [ ] I can explain features, labels, training and prediction
- [ ] I know regression vs classification
- [ ] I can clean data (zeros as missing values) with NumPy
- [ ] I can split data into train/test and explain why
- [ ] I can standardize features without leaking test information
- [ ] I implemented linear regression (normal equation and gradient descent)
- [ ] I implemented logistic regression with the sigmoid and log loss
- [ ] I implemented k-NN with broadcasting
- [ ] I can compute accuracy, precision, recall and F1, and explain why accuracy can be misleading
- [ ] I can explain overfitting in one sentence
- [ ] I wrapped a model into a class with `fit` / `predict` / `score`

## 📚 Where to go next
1. [Kaggle Learn: Intro to ML](https://www.kaggle.com/learn/intro-to-machine-learning) and [Pandas](https://www.kaggle.com/learn/pandas)
2. [scikit-learn Getting Started](https://scikit-learn.org/stable/getting_started.html)
3. [pandas: 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
4. Your first Kaggle competition: [Titanic: Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic)
5. Neural networks: [Neural Networks and Deep Learning (free book)](http://neuralnetworksanddeeplearning.com/) or the [PyTorch tutorials](https://pytorch.org/tutorials/)

🎓 **Congratulations!** In four weeks you went from `print("Hello")` to training models from scratch. Keep building, keep reading the docs and keep committing.